## Task 2 — Clean + Enrich Data (`data/` read-only)

Cleans and enriches transactions for a single `client_id` by:
- parsing currency + datetimes
- normalizing ZIP and MCC
- deduping by transaction `id`
- joining non-PII fields from `users.csv` and `cards.csv`
- adding `mcc_description` from `mcc_codes.json`

Outputs (flat files under `artifacts/`):
- `artifacts/transactions_enriched_{client_id}.json`
- `artifacts/qa_report_{client_id}.json`

PII policy (per `PROJECT_SPEC.md`): exclude `address`, `card_number`, and `cvv` from outputs.


## Install dependencies (run once)

If you're running in a fresh environment, install required packages from `requirements.txt`.

In [9]:
!python3 -m pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## Imports

Uses `pandas` for chunked CSV reads and Parquet output.

In [10]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd


## Config

Default demo user is `1696` per spec. Transactions are processed in chunks to avoid loading the full dataset.

In [11]:
CLIENT_ID = 1696
TRANSACTIONS_CHUNKSIZE = 250_000
CSV_SAMPLE_ROWS_FOR_QA = 5
PREVIEW_ROWS = 5


def find_project_root(start: Path | None = None) -> Path:
    """Find project root by locating the `data/` directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing a 'data/' directory")


ROOT = find_project_root()
DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"

PATHS = {
    "transactions": DATA_DIR / "transactions.csv",
    "cards": DATA_DIR / "cards.csv",
    "users": DATA_DIR / "users.csv",
    "mcc": DATA_DIR / "mcc_codes.json",
}

OUT_ENRICHED_JSON = ARTIFACTS_DIR / f"transactions_enriched_{CLIENT_ID}.json"
OUT_QA = ARTIFACTS_DIR / f"qa_report_{CLIENT_ID}.json"

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
for k, v in PATHS.items():
    print(f"{k} -> {v}")


ROOT: /Users/nicholasp/Personal Coding/JHU/personal finance
DATA_DIR: /Users/nicholasp/Personal Coding/JHU/personal finance/data
ARTIFACTS_DIR: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts
transactions -> /Users/nicholasp/Personal Coding/JHU/personal finance/data/transactions.csv
cards -> /Users/nicholasp/Personal Coding/JHU/personal finance/data/cards.csv
users -> /Users/nicholasp/Personal Coding/JHU/personal finance/data/users.csv
mcc -> /Users/nicholasp/Personal Coding/JHU/personal finance/data/mcc_codes.json


## Helpers

Small functions that are easy to test and reuse later in feature engineering.

What gets cleaned/standardized:
- `amount` (currency string) → `amount_usd` (float)
- `date` (string) → `transaction_dt` (datetime, then serialized to ISO string for JSON)
- `zip` → `zip_norm` (string, strips `.0`)
- `mcc` → `mcc_code` (digit string) and `mcc_description` (lookup, else `UNKNOWN_MCC`)
- `use_chip` + `merchant_city` → `is_online` (boolean heuristic)
- Deduping: drop duplicate transaction `id` across chunks and within-chunk
- PII removal: drop `address`, `card_number`, `cvv` from all outputs


In [12]:
_CURRENCY_STRIP_RE = re.compile(r"[^0-9\-\.]" )


def clean_text(value: Any) -> str | None:
    """Strip whitespace and collapse internal spaces; returns None for empty/NA."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if not s:
        return None
    # collapse internal whitespace
    s = re.sub(r"\s+", " ", s)
    return s if s else None


def parse_currency_to_float(value: Any) -> float | None:
    """Parse values like "$2,238 " into float. Returns None on failure."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if s == "":
        return None
    s2 = _CURRENCY_STRIP_RE.sub("", s)
    if s2 in ("", "-", "."):
        return None
    try:
        return float(s2)
    except ValueError:
        return None


def parse_transaction_datetime(value: Any) -> datetime | None:
    """Parse transaction datetime like '2010-01-01 00:07:00'."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = clean_text(value)
    if s is None:
        return None
    try:
        return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None


def normalize_zip(value: Any) -> str | None:
    """Normalize ZIP-like values: '10464.0' -> '10464'."""
    s = clean_text(value)
    if s is None:
        return None
    if s.endswith(".0"):
        s = s[:-2]
    return s


def normalize_mcc(value: Any) -> str | None:
    """Normalize MCC values to digit strings: 5812, '5812.0' -> '5812'."""
    s = clean_text(value)
    if s is None:
        return None
    if s.endswith(".0"):
        s = s[:-2]
    s = s.strip()
    return s if s.isdigit() else None


def normalize_state(value: Any) -> str | None:
    """Normalize state codes to uppercase; returns None if missing."""
    s = clean_text(value)
    if s is None:
        return None
    return s.upper()


def yesno_to_bool(value: Any) -> bool | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip().lower()
    if s in ("yes", "y", "true", "1"):
        return True
    if s in ("no", "n", "false", "0"):
        return False
    return None


def derive_is_online(use_chip: Any, merchant_city: Any) -> bool:
    uc = str(use_chip).lower() if use_chip is not None else ""
    mc = str(merchant_city).lower() if merchant_city is not None else ""
    return ("online" in uc) or (mc == "online")


def dedupe_by_id(df: pd.DataFrame, *, id_col: str, seen_ids: set[str]) -> tuple[pd.DataFrame, int]:
    """Drop duplicate IDs across chunks; returns (deduped_df, dropped_count)."""
    ids = df[id_col].astype(str)
    is_dup = ids.isin(seen_ids)
    dropped = int(is_dup.sum())
    if dropped:
        df = df.loc[~is_dup].copy()
        ids = df[id_col].astype(str)
    # also drop duplicates within the chunk
    before = len(df)
    df = df.drop_duplicates(subset=[id_col], keep="first")
    dropped += before - len(df)
    seen_ids.update(ids.tolist())
    return df, dropped


@dataclass
class QaReport:
    client_id: int
    transactions_rows_total_scanned: int = 0
    transactions_rows_for_client: int = 0
    transactions_duplicates_dropped: int = 0
    amount_parse_failures: int = 0
    date_parse_failures: int = 0
    rows_dropped_invalid_amount: int = 0
    rows_dropped_invalid_date: int = 0
    card_join_misses: int = 0
    mcc_missing_from_lookup: int = 0
    sample_transactions_head: list[dict[str, Any]] | None = None


## Load dimensions (`users`, `cards`, `mcc`) (PII-safe)

Loads small dimension tables fully and keeps only non-PII columns.

In [13]:
for k, p in PATHS.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

mcc_lookup: dict[str, str] = json.loads(PATHS["mcc"].read_text(encoding="utf-8"))

users = pd.read_csv(PATHS["users"], dtype=str)
user_row = users.loc[users["id"].astype(str) == str(CLIENT_ID)]
if user_row.empty:
    raise ValueError(f"client_id {CLIENT_ID} not found in users.csv")
user_row = user_row.iloc[0].to_dict()

monthly_limit = parse_currency_to_float(user_row.get("monthly_discretionary_limits"))
if monthly_limit is None:
    raise ValueError("monthly_discretionary_limits is missing/unparseable for this client")

user_features = {
    "client_id": int(CLIENT_ID),
    "monthly_discretionary_limit_usd": monthly_limit,
    "current_age": user_row.get("current_age"),
    "retirement_age": user_row.get("retirement_age"),
    "birth_year": user_row.get("birth_year"),
    "birth_month": user_row.get("birth_month"),
    "gender": user_row.get("gender"),
    "latitude": user_row.get("latitude"),
    "longitude": user_row.get("longitude"),
    "per_capita_income": user_row.get("per_capita_income"),
    "yearly_income": user_row.get("yearly_income"),
    "total_debt": user_row.get("total_debt"),
    "credit_score": user_row.get("credit_score"),
    "num_credit_cards": user_row.get("num_credit_cards"),
}

cards = pd.read_csv(PATHS["cards"], dtype=str)
cards = cards.loc[cards["client_id"].astype(str) == str(CLIENT_ID)].copy()
# Drop PII from cards immediately
cards = cards.drop(columns=[c for c in ["card_number", "cvv"] if c in cards.columns])
cards["has_chip"] = cards["has_chip"].map(yesno_to_bool)
cards["card_on_dark_web"] = cards["card_on_dark_web"].map(yesno_to_bool)
cards["credit_limit_usd"] = cards["credit_limit"].map(parse_currency_to_float)
cards = cards.drop(columns=[c for c in ["credit_limit"] if c in cards.columns])

print("Loaded cards for client:", len(cards))


Loaded cards for client: 1


## Clean + filter transactions (chunked)

Processes `transactions.csv` in chunks and keeps only rows for the selected `CLIENT_ID`.

In [14]:
qa = QaReport(client_id=CLIENT_ID)
seen_tx_ids: set[str] = set()
frames: list[pd.DataFrame] = []

usecols = [
    "id",
    "date",
    "client_id",
    "card_id",
    "amount",
    "use_chip",
    "merchant_id",
    "merchant_city",
    "merchant_state",
    "zip",
    "mcc",
    "errors",
]

for chunk in pd.read_csv(PATHS["transactions"], dtype=str, usecols=usecols, chunksize=TRANSACTIONS_CHUNKSIZE):
    qa.transactions_rows_total_scanned += len(chunk)
    chunk = chunk.loc[chunk["client_id"].astype(str) == str(CLIENT_ID)].copy()
    if chunk.empty:
        continue
    qa.transactions_rows_for_client += len(chunk)

    # Standardize text fields (trim/collapse whitespace)
    for col in [
        "id",
        "date",
        "client_id",
        "card_id",
        "amount",
        "use_chip",
        "merchant_id",
        "merchant_city",
        "merchant_state",
        "zip",
        "mcc",
        "errors",
    ]:
        if col in chunk.columns:
            chunk[col] = chunk[col].map(clean_text)

    chunk["merchant_state"] = chunk["merchant_state"].map(normalize_state)


    chunk, dropped = dedupe_by_id(chunk, id_col="id", seen_ids=seen_tx_ids)
    qa.transactions_duplicates_dropped += dropped

    # parse/normalize
    chunk["amount_usd"] = chunk["amount"].map(parse_currency_to_float)
    qa.amount_parse_failures += int(chunk["amount_usd"].isna().sum())

    # Drop rows with unparseable amount (inconsistent records)
    invalid_amount = chunk["amount_usd"].isna()
    qa.rows_dropped_invalid_amount += int(invalid_amount.sum())
    if int(invalid_amount.sum()) > 0:
        chunk = chunk.loc[~invalid_amount].copy()


    chunk["transaction_dt"] = chunk["date"].map(parse_transaction_datetime)
    qa.date_parse_failures += int(pd.isna(chunk["transaction_dt"]).sum())

    # Drop rows with unparseable datetime (inconsistent records)
    invalid_date = pd.isna(chunk["transaction_dt"])
    qa.rows_dropped_invalid_date += int(invalid_date.sum())
    if int(invalid_date.sum()) > 0:
        chunk = chunk.loc[~invalid_date].copy()


    chunk["zip_norm"] = chunk["zip"].map(normalize_zip)
    chunk["mcc_code"] = chunk["mcc"].map(normalize_mcc)
    chunk["mcc_description"] = chunk["mcc_code"].map(lambda c: mcc_lookup.get(str(c), "UNKNOWN_MCC") if c else "UNKNOWN_MCC")
    qa.mcc_missing_from_lookup += int((chunk["mcc_description"] == "UNKNOWN_MCC").sum())

    chunk["is_online"] = chunk.apply(lambda r: derive_is_online(r.get("use_chip"), r.get("merchant_city")), axis=1)

    frames.append(chunk)

transactions = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=usecols)
print("Transactions for client after filtering:", len(transactions))
if len(transactions):
    qa.sample_transactions_head = transactions.head(CSV_SAMPLE_ROWS_FOR_QA).to_dict(orient="records")


Transactions for client after filtering: 30672


## Join + write artifacts

Joins non-PII card/user fields and writes flat artifacts under `artifacts/`.

Note: output is JSON (not Parquet) per your latest preference.

In [15]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Join cards on card_id -> cards.id
transactions["card_id"] = transactions["card_id"].astype(str)
cards_for_join = cards.rename(columns={"id": "card_id"}).copy()
cards_for_join["card_id"] = cards_for_join["card_id"].astype(str)

enriched = transactions.merge(cards_for_join, on="card_id", how="left", suffixes=("", "_card"))
qa.card_join_misses = int(enriched["card_brand"].isna().sum()) if "card_brand" in enriched.columns else 0

# Add user features (excluding address) as constant columns
for k, v in user_features.items():
    if k == "client_id":
        continue
    enriched[k] = v

# Ensure PII columns are not present
for pii in ["address", "card_number", "cvv"]:
    if pii in enriched.columns:
        enriched = enriched.drop(columns=[pii])

# Serialize datetimes to ISO strings for JSON output
if "transaction_dt" in enriched.columns:
    enriched["transaction_dt"] = enriched["transaction_dt"].map(
        lambda d: d.isoformat(sep=" ") if d is not None and not pd.isna(d) else None
    )

# Write outputs (flat files)
enriched.to_json(OUT_ENRICHED_JSON, orient="records", indent=2)
OUT_QA.write_text(
    json.dumps(asdict(qa), indent=2, sort_keys=True, default=str),
    encoding="utf-8",
)

print("Wrote:", OUT_ENRICHED_JSON)
print("Wrote:", OUT_QA)
display(enriched.head(PREVIEW_ROWS))

Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/transactions_enriched_1696.json
Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/qa_report_1696.json


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,birth_year,birth_month,gender,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,7475539,2010-01-01 05:38:00,1696,2408,$4.02,Swipe Transaction,35451,Merritt Island,FL,32952.0,...,1956,12,Female,28.32,-80.68,"$26,339","$53,702","$85,160",606,1
1,7475586,2010-01-01 06:03:00,1696,2408,$9.68,Online Transaction,39021,ONLINE,None,None,...,1956,12,Female,28.32,-80.68,"$26,339","$53,702","$85,160",606,1
2,7475755,2010-01-01 06:53:00,1696,2408,$3.41,Swipe Transaction,75781,Merritt Island,FL,32953.0,...,1956,12,Female,28.32,-80.68,"$26,339","$53,702","$85,160",606,1
3,7477220,2010-01-01 12:18:00,1696,2408,$9.94,Online Transaction,50404,ONLINE,None,None,...,1956,12,Female,28.32,-80.68,"$26,339","$53,702","$85,160",606,1
4,7477493,2010-01-01 13:11:00,1696,2408,$-89.00,Swipe Transaction,61195,Merritt Island,FL,32952.0,...,1956,12,Female,28.32,-80.68,"$26,339","$53,702","$85,160",606,1
